# Аналіз структури та якості датасету

## Магістерська кваліфікаційна робота

**Тема:**

Прогнозування фізіологічних та поведінкових параметрів користувачів фітнес-трекерів методами машинного навчання.

---

### Мета

Дослідити структуру набору даних, оцінити його якість та підтвердити можливість використання для подальшого машинного навчання.

In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import sys
import importlib
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# LOAD PROJECT MODULES
# ============================================================

PROJECT_PATH = "/content/drive/MyDrive/FitnessML_Master"

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

importlib.invalidate_caches()

import config
import utils

config = importlib.reload(config)
utils = importlib.reload(utils)

utils.section("Project modules")

print("✓ config.py loaded")
print("✓ utils.py loaded")


PROJECT MODULES
✓ config.py loaded
✓ utils.py loaded


In [12]:
# ============================================================
# LOAD DATASET
# ============================================================

df = utils.load_dataset()
utils.set_plot_style()
utils.section("Dataset")

print("✓ Dataset loaded")
print(config.DATASET_NAME)


DATASET
✓ Dataset loaded
health_fitness_tracking_365days.csv


## Первинний огляд датасету

In [5]:
# ============================================================
# DATASET OVERVIEW
# ============================================================

utils.section("Dataset overview")

print(f"Rows      : {df.shape[0]:,}")
print(f"Columns   : {df.shape[1]}")
print(f"Memory    : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")


DATASET OVERVIEW
Rows      : 365,000
Columns   : 12
Memory    : 65.79 MB


In [13]:
utils.subsection("Random sample")

display(
    df.sample(
        5,
        random_state=config.RANDOM_STATE
    )
)


----------------------------------------
Random sample
----------------------------------------


,user_id,age,gender,date,steps,heart_rate_avg,sleep_hours,calories_burned,exercise_minutes,stress_level,weight_kg,bmi
85602,234,32,M,2025-03-17,7684,46.228031,5.309158,2035.456885,16.812398,6,90.003313,21.128324
209726,574,62,F,2025-04-10,9175,64.089593,9.126044,1942.130441,15.554318,1,45.269551,17.417845
273099,748,24,M,2024-11-24,10935,68.876408,5.342251,1980.777403,17.696382,6,70.816581,17.609043
218614,598,27,M,2025-08-16,6463,81.567042,7.557913,1975.235772,42.255248,8,52.919795,22.159854
326188,893,64,F,2025-05-07,7682,46.464368,5.802996,2083.172953,37.037910,7,49.858163,29.797799


In [15]:
utils.subsection("Data types")
types = df.dtypes.to_frame(name="Data type")

display(
    utils.rename_columns(types)
)


----------------------------------------
Data types
----------------------------------------


,Data type
user_id,int64
age,int64
gender,object
date,object
steps,int64
heart_rate_avg,float64
sleep_hours,float64
calories_burned,float64
exercise_minutes,float64
stress_level,int64


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365000 entries, 0 to 364999
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_id           365000 non-null  int64  
 1   age               365000 non-null  int64  
 2   gender            365000 non-null  object 
 3   date              365000 non-null  object 
 4   steps             365000 non-null  int64  
 5   heart_rate_avg    365000 non-null  float64
 6   sleep_hours       365000 non-null  float64
 7   calories_burned   365000 non-null  float64
 8   exercise_minutes  365000 non-null  float64
 9   stress_level      365000 non-null  int64  
 10  weight_kg         365000 non-null  float64
 11  bmi               365000 non-null  float64
dtypes: float64(6), int64(4), object(2)
memory usage: 33.4+ MB


## Оцінка якості даних

In [18]:
utils.section("Data quality")
utils.subsection("Missing values")
missing = df.isnull().sum().to_frame(name="Missing values")

missing["Percent"] = (
    missing["Missing values"] / len(df) * 100
).round(2)

display(missing)


DATA QUALITY

----------------------------------------
Missing values
----------------------------------------


,Missing values,Percent
user_id,0,0.0
age,0,0.0
gender,0,0.0
date,0,0.0
steps,0,0.0
heart_rate_avg,0,0.0
sleep_hours,0,0.0
calories_burned,0,0.0
exercise_minutes,0,0.0
stress_level,0,0.0


In [ ]:
duplicates = df.duplicated().sum()

print(f"Duplicate rows : {duplicates}")

Duplicate rows : 0


In [ ]:
print(f"Unique users : {df['user_id'].nunique():,}")
print(f"Average records/user : {len(df)/df['user_id'].nunique():.0f}")

Unique users : 1,000
Average records/user : 365


In [ ]:
df["date"] = pd.to_datetime(df["date"])

print(df["date"].min())
print(df["date"].max())
print(df["date"].nunique())

2024-09-06 00:00:00
2025-09-05 00:00:00
365


In [17]:
utils.subsection("Descriptive statistics")
describe = df.describe().T

display(
    utils.rename_columns(describe)
)


----------------------------------------
Descriptive statistics
----------------------------------------


,count,mean,std,min,25%,50%,75%,max
user_id,365000.0,499.500000,288.675386,0.000000,249.750000,499.500000,749.250000,999.000000
age,365000.0,48.772000,17.792832,18.000000,33.000000,49.000000,64.000000,79.000000
steps,365000.0,8499.532058,1446.826636,5723.000000,7248.000000,8497.000000,9750.000000,11330.000000
heart_rate_avg,365000.0,64.979414,11.246659,13.801607,57.390843,64.997519,72.550623,113.325619
sleep_hours,365000.0,7.003109,1.495827,3.000000,5.990209,7.000849,8.015041,12.000000
calories_burned,365000.0,1999.942722,300.058450,505.956221,1797.603287,1999.980702,2201.965489,3408.341803
exercise_minutes,365000.0,30.054553,30.145841,0.000025,8.600785,20.784057,41.664980,487.105147
stress_level,365000.0,5.501970,2.872646,1.000000,3.000000,6.000000,8.000000,10.000000
weight_kg,365000.0,70.011188,15.005358,4.327270,59.900270,69.995579,80.138985,138.765278
bmi,365000.0,21.998630,4.007091,4.354979,19.295143,21.997430,24.692963,40.445029


In [9]:
# ============================================================
# EXPORT RESULTS
# ============================================================

utils.save_table_ua(
    describe,
    "01_describe.csv"
)

utils.save_table_ua(
    missing,
    "01_missing_values.csv"
)

utils.save_table_ua(
    types,
    "01_data_types.csv"
)

✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/01_describe.csv
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/01_missing_values.csv
✓ Table saved -> /content/drive/MyDrive/FitnessML_Master/tables/01_data_types.csv


In [ ]:
report = f"""
# Dataset Audit

## Dataset

Records: {len(df):,}

Features: {df.shape[1]}

Users: {df['user_id'].nunique():,}

Observation period:
{df['date'].min().date()} — {df['date'].max().date()}

## Data Quality

Missing values: {df.isnull().sum().sum()}

Duplicate rows: {duplicates}

## Conclusion

The dataset is complete and suitable for exploratory data analysis
and machine learning experiments.
"""

utils.save_report(report, "01_dataset_audit.md")

✓ Report saved -> /content/drive/MyDrive/FitnessML_Master/reports/01_dataset_audit.md


In [10]:
utils.section("Dataset audit completed")


DATASET AUDIT COMPLETED
